# Survey inter-rater reliability

Reliability uses original independent ratings before adjudication for the eligible cohorts: post-submission n=12 and pre-submission n=4. Processed data and the complete per-code audit retain 102 and 39 observed coded answers across questions, including pre-submission Q26. Reported reliability excludes pre Q26, matching its exclusion from substantive analysis: 102 post and 36 pre answers, across 11 and ten coded questions respectively. The locked 128-code data remain unchanged; 123 codes are included in reporting.

For each question, calculate Cohen’s kappa separately for every code, then take the median of the estimable code kappas. Retain the number of estimable codes, IQR and range; undefined kappas remain missing, never zero. Calculate Gwet’s AC1 summaries in parallel. These question-level medians are distinct from the separate survey-level median and IQR of question medians, which weight questions equally. No pooled coefficient or pass/fail threshold is used. Quantiles use linear interpolation.

`survey_irr_per_code.csv` is the complete audit: `included_in_report` identifies the reported subset. The displayed per-code results and both question/survey summary CSVs exclude pre Q26, while its data and audit values are retained.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data/processed"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
counts = pd.read_csv(DATA / "survey_irr_counts.csv")
book = pd.read_csv(DATA / "survey_codebook.csv")
KEYS = ["survey_id", "question_id", "code_id"]
CELLS = ["both_present", "a_only", "b_only", "both_absent"]
assert not counts.duplicated(KEYS).any() and counts.cohort.eq("eligible_analysis").all()
assert counts[CELLS].ge(0).all().all()
assert np.equal(counts[CELLS], np.floor(counts[CELLS])).all().all()

# All calculations use the original independent decisions, before adjudication.
per_code = counts.merge(book[KEYS+["code_label"]], on=KEYS, validate="one_to_one")
a, b, c, d = (per_code[name].astype(float) for name in CELLS)
n = a+b+c+d
assert n.gt(0).all()
observed = (a+d)/n
p_a, p_b = (a+b)/n, (a+c)/n
chance_kappa = p_a*p_b + (1-p_a)*(1-p_b)
p_bar = (p_a+p_b)/2
chance_ac1 = 2*p_bar*(1-p_bar)
per_code["n_units"] = n.astype(int)
per_code["observed_agreement"] = observed
per_code["positive_agreement"] = 2*a/(2*a+b+c).replace(0, np.nan)
per_code["negative_agreement"] = 2*d/(2*d+b+c).replace(0, np.nan)
per_code["cohens_kappa"] = (observed-chance_kappa)/(1-chance_kappa).where(~np.isclose(chance_kappa, 1))
per_code["gwets_ac1"] = (observed-chance_ac1)/(1-chance_ac1)
per_code["kappa_status"] = np.where(per_code.cohens_kappa.isna(), "not_estimable_constant_ratings", "estimated")
# Analysis-only exclusion: retain Q26 in data and audit, omit it from reported reliability.
per_code["included_in_report"] = ~((per_code.survey_id == "presub") & (per_code.question_id == 26))
reported_codes = per_code.loc[per_code.included_in_report].copy()
assert len(per_code) == 128 and len(reported_codes) == 123
display(reported_codes)


## Median reliability per question and separate survey summaries

Each question median is calculated across its estimable code coefficients. The code counts show how many contributed.

Post-submission Q17 and pre-submission Q20 each contain one observed coded answer, with complete agreement between the independent coders. Both coders assigned both Q17 codes and the single Q20 code. Observed and chance-expected agreement are both one, so Cohen’s kappa is undefined; no question-level median kappa can be calculated. Retain the findings and report: “Kappa not estimable; one coded answer, with complete observed agreement.” AC1 equals one, but agreement on one answer provides very limited evidence about coding consistency.


In [ ]:
question_rows, survey_rows = [], []
for metric in ["cohens_kappa", "gwets_ac1"]:
    for (survey, question), group in reported_codes.groupby(["survey_id", "question_id"]):
        values = group[metric].dropna()
        question_rows.append({"survey_id": survey, "question_id": question, "metric": metric,
            "codes_total": len(group), "codes_estimable": len(values), "median": values.median(),
            "q1": values.quantile(0.25, interpolation="linear"), "q3": values.quantile(0.75, interpolation="linear"),
            "minimum": values.min(), "maximum": values.max()})
question_summary = pd.DataFrame(question_rows)
for (survey, metric), group in question_summary.groupby(["survey_id", "metric"]):
    values = group["median"].dropna()
    survey_rows.append({"survey_id": survey, "metric": metric, "questions_total": len(group),
        "questions_estimable": len(values), "median_of_question_medians": values.median(),
        "q1": values.quantile(0.25, interpolation="linear"), "q3": values.quantile(0.75, interpolation="linear")})
survey_summary = pd.DataFrame(survey_rows)
for metric, label in [("cohens_kappa", "Median Cohen’s kappa per question"), ("gwets_ac1", "Median Gwet’s AC1 per question")]:
    print(label)
    display(question_summary.loc[question_summary.metric == metric].drop(columns="metric"))
print("Separate survey-level summaries of question medians")
display(survey_summary)
per_code.to_csv(OUTPUT / "survey_irr_per_code.csv", index=False)
question_summary.to_csv(OUTPUT / "survey_irr_by_question.csv", index=False)
survey_summary.to_csv(OUTPUT / "survey_irr_by_survey.csv", index=False)
